# MemoryNPC Validation Lab

This notebook is a technical validation companion to `notebook.ipynb`. The main report explains the assignment and design choices. This lab focuses on pressure-testing the NLP pipeline itself.

The goal is not to make the fantasy narrative longer. The goal is to inspect whether the language-processing components are doing useful work: intent detection, memory extraction, semantic retrieval, trust-state tracking, grounded response generation, and validation logging.

## What This Lab Tests

This notebook checks the pressure points of the project:

- Does the agent classify player intent correctly?
- Does it store durable memories and ignore filler?
- Can FAISS retrieve memories when the query uses different wording?
- Does the trust score change deterministically?
- Does trust change when the player follows or ignores advice?
- Do trust thresholds actually influence generated responses?
- Does the agent avoid unsupported memory claims?
- Is the validation trace detailed enough to explain what happened?

This is also a notebook-based mini app: use the `chat()` helper to send messages and inspect the trace without opening Streamlit.

## Setup

The lab uses the same backend as the Streamlit app. This is important because validation should test the real system, not a separate copy of the logic.

In [ ]:
# Standard tools for displaying validation tables.
import os
import pandas as pd
from dotenv import load_dotenv
from IPython.display import display, Markdown

# Import the real project backend. The notebook acts like an app shell around it.
from npc_agent import MemoryNPC

# Load the OpenAI key without printing it.
load_dotenv()
print('API key loaded:', bool(os.getenv('OPENAI_API_KEY')))

## Notebook Mini App

The cell below creates a small notebook interface. Change `player_message` and rerun the cell to continue the conversation. After each turn, the notebook shows the response, memory table, and validation trace.

In [ ]:
# One persistent agent for notebook interaction.
notebook_agent = MemoryNPC()

def show_agent_state(agent):
    """Display the same internal state that the Streamlit sidebar shows."""
    display(Markdown(f'**Trust score:** {agent.trust_score}'))
    display(Markdown(f'**Trust level:** {agent.get_trust_level()}'))
    display(Markdown('**Memory table**'))
    display(agent.get_memory_table())
    display(Markdown('**Validation trace**'))
    display(agent.get_event_log())

def chat(message):
    """Notebook-app chat function: send one message and inspect the pipeline output."""
    result = notebook_agent.generate_npc_response(message)
    display(Markdown(f'**Player:** {message}'))
    display(Markdown(f'**Elara:** {result["npc_response"]}'))
    display(pd.DataFrame([result]))
    show_agent_state(notebook_agent)
    return result

In [ ]:
# Change this message and rerun this cell to use the notebook like a small app.
player_message = 'My name is Matt'
chat(player_message)

## Helper: Run A Scripted Conversation

Scripted conversations are useful because they make the evaluation repeatable. Instead of only showing one hand-picked answer, we can run the same sequence and inspect the event log.

In [ ]:
def run_script(messages, starting_trust=50):
    """Run a list of player messages and return the agent plus its validation trace."""
    agent = MemoryNPC()
    agent.trust_score = starting_trust
    rows = []
    for message in messages:
        rows.append(agent.generate_npc_response(message))
    return agent, pd.DataFrame(rows)

script = [
    'Hello Elara',
    'My name is Matt',
    'I lost my sword near the forest',
    'Do you remember what I lost?',
    'You are a very skilled blacksmith',
    'You are useless',
    'Can you still help me?',
]

script_agent, script_trace = run_script(script)
script_trace

In [ ]:
# The memory table should show durable facts and relationship events,
# not every single line of dialogue.
script_agent.get_memory_table()

## Pressure Test 1: Intent Detection

This tests the NLP interpretation step. A response can sound good even if the system misunderstood the player's intent, so this should be evaluated separately.

In [ ]:
intent_cases = [
    ('Hello Elara', 'greeting'),
    ('Can you give me a quest?', 'ask_quest'),
    ('My name is Matt', 'share_fact'),
    ('Do you remember what I lost?', 'ask_memory'),
    ('You are a skilled blacksmith', 'compliment'),
    ('You are useless', 'insult'),
    ('Can you help me fix my shield?', 'ask_help'),
    ('Goodbye', 'goodbye'),
    ('The sky is cloudy', 'unknown'),
]

intent_agent = MemoryNPC()
intent_rows = []
for text, expected in intent_cases:
    predicted = intent_agent.classify_intent(text)
    intent_rows.append({
        'input': text,
        'expected': expected,
        'predicted': predicted,
        'correct': predicted == expected,
    })

intent_df = pd.DataFrame(intent_rows)
display(intent_df)
print('Intent accuracy:', f'{intent_df["correct"].mean():.0%}')

## Pressure Test 2: Memory Extraction Selectivity

The memory extractor should save durable facts, not filler. This matters because a noisy memory store makes retrieval worse.

In [ ]:
memory_extraction_cases = [
    'Hello there',
    'My name is Matt',
    'I lost my sword near the forest',
    'You are a very skilled blacksmith',
    'The weather is kind of gray today',
]

memory_agent = MemoryNPC()
memory_rows = []
for text in memory_extraction_cases:
    intent = memory_agent.classify_intent(text)
    extracted = memory_agent.extract_memory(text)
    memory_rows.append({
        'input': text,
        'intent': intent,
        'extracted_memory': extracted,
        'saved_by_rule': extracted.upper() != 'NONE',
    })

pd.DataFrame(memory_rows)

## Pressure Test 3: Semantic Retrieval Versus Keyword Baseline

This tests the central memory claim. FAISS should retrieve semantically related memories even when the query does not use identical wording.

In [ ]:
known_memories = [
    ('lost_sword', 'The player lost a sword near the forest.'),
    ('likes_axes', 'The player prefers heavy axes over light daggers.'),
    ('brother_goal', 'The player wants to find their missing brother.'),
    ('praised_elara', 'The player praised Elara as a skilled blacksmith.'),
]

retrieval_queries = [
    ('What weapon did I misplace near the trees?', 'sword'),
    ('What type of weapon do I prefer?', 'axes'),
    ('Who am I searching for?', 'brother'),
    ('Did I compliment the blacksmith?', 'skilled'),
]

retrieval_agent = MemoryNPC()
for memory_type, text in known_memories:
    retrieval_agent.add_memory(text, memory_type=memory_type, importance=2)

def keyword_retrieve(query, memories, k=3):
    """Simple baseline: rank memories by shared words only."""
    query_words = set(query.lower().replace('?', '').replace('.', '').split())
    scored = []
    for memory_type, text in memories:
        memory_words = set(text.lower().replace('.', '').split())
        scored.append((len(query_words & memory_words), text))
    scored.sort(reverse=True)
    return [text for score, text in scored[:k] if score > 0]

retrieval_rows = []
for query, expected_keyword in retrieval_queries:
    faiss_hits = [m['text'] for m in retrieval_agent.retrieve_memories(query, k=3)]
    keyword_hits = keyword_retrieve(query, known_memories, k=3)
    retrieval_rows.append({
        'query': query,
        'expected_keyword': expected_keyword,
        'faiss_top_3': faiss_hits,
        'faiss_success': any(expected_keyword.lower() in text.lower() for text in faiss_hits),
        'keyword_top_3': keyword_hits,
        'keyword_success': any(expected_keyword.lower() in text.lower() for text in keyword_hits),
    })

retrieval_pressure_df = pd.DataFrame(retrieval_rows)
display(retrieval_pressure_df)
print('FAISS success:', f'{retrieval_pressure_df["faiss_success"].mean():.0%}')
print('Keyword baseline success:', f'{retrieval_pressure_df["keyword_success"].mean():.0%}')

## Pressure Test 4: Trust Thresholds And Response Tone

This test asks the same help request at different trust levels. The point is not to prove perfect emotional realism. The point is to check whether the deterministic trust state is actually injected into the generation step.

In [ ]:
trust_pressure_rows = []
for starting_score in [25, 50, 85]:
    trust_agent = MemoryNPC()
    trust_agent.trust_score = starting_score
    trust_agent.add_memory('The player lost a sword near the forest.', memory_type='fact', importance=2)
    result = trust_agent.generate_npc_response('Can you help me recover my sword?')
    trust_pressure_rows.append({
        'starting_trust': starting_score,
        'trust_level': result['trust_level'],
        'intent': result['intent'],
        'retrieved_memories': result['retrieved_memories'],
        'response': result['npc_response'],
    })

pd.DataFrame(trust_pressure_rows)

## Pressure Test 5: Advice Following And Defiance

This test makes trust depend on behavior. Elara first gives advice. Then the player either does the opposite or follows the advice. The validation trace should show a negative `advice_delta` for ignoring advice and a positive `advice_delta` for following it.

In [ ]:
advice_agent = MemoryNPC()
advice_messages = [
    'Can you help me recover my sword?',
    'I will go alone anyway',
    'Can you help me recover my sword?',
    'I will follow your advice and bring a torch',
]

advice_rows = []
for message in advice_messages:
    advice_rows.append(advice_agent.generate_npc_response(message))

pd.DataFrame(advice_rows)[[
    'turn', 'player_input', 'intent', 'trust_before', 'trust_score',
    'trust_delta', 'advice_status', 'advice_delta', 'active_advice', 'new_advice', 'npc_response'
]]

## Pressure Test 6: Unsupported Memory Claims

A major risk in LLM systems is hallucinated memory. This test asks Elara about something that has not been stored. The response should not invent a detailed memory.

In [ ]:
unsupported_agent = MemoryNPC()
unsupported_result = unsupported_agent.generate_npc_response('Do you remember the name of my brother?')

pd.DataFrame([{
    'player_input': unsupported_result['player_input'],
    'intent': unsupported_result['intent'],
    'retrieved_memories': unsupported_result['retrieved_memories'],
    'response': unsupported_result['npc_response'],
    'manual_check': 'Response should not invent a brother name because no memory was retrieved.',
}])

## Pressure Test 7: Validation Trace Completeness

This checks whether the system stores enough data to explain a response after the fact. A good trace should include the original input, intent, memory actions, retrieval results, trust changes, and response.

In [ ]:
required_trace_columns = {
    'turn',
    'player_input',
    'intent',
    'extracted_memory',
    'saved_memory',
    'retrieved_memories',
    'trust_before',
    'trust_score',
    'trust_level',
    'trust_delta',
    'advice_status',
    'advice_delta',
    'active_advice',
    'new_advice',
    'npc_response',
}

trace_columns = set(script_agent.get_event_log().columns)
missing_columns = sorted(required_trace_columns - trace_columns)
pd.DataFrame([{
    'required_columns': len(required_trace_columns),
    'actual_columns': len(trace_columns),
    'missing_columns': missing_columns,
    'trace_complete': len(missing_columns) == 0,
}])

## Summary: What Counts As Good Performance?

For this project, good performance means:

- Intent detection is accurate on manually designed cases.
- Durable memory extraction saves important facts and avoids filler.
- FAISS retrieval reaches the target of at least 80% top-3 success.
- FAISS retrieval performs at least as well as the keyword baseline, especially on paraphrased queries.
- Trust score changes are deterministic and visible in the trace.
- Following advice increases trust and explicitly ignoring advice decreases trust.
- Trust thresholds produce visibly different response tones.
- The NPC avoids unsupported memory claims when no relevant memory is retrieved.
- The event log is complete enough to explain the final answer.

This keeps the evaluation focused on the NLP architecture rather than only the fantasy dialogue.